# Prompt Engineering to Agentic AI: Hands-On Activity

## Building a Restaurant Assistant, One Technique at a Time

In this notebook, you'll implement each prompting technique from the lecture using a restaurant/cooking theme. Each section builds on the last — by the end, you'll have a working AI agent that can reason about recipes, look up a knowledge base, and take actions.

**Prerequisites:** An OpenAI API key set as the environment variable `OPENAI_API_KEY`.

---
## Setup

Run the cells below to install dependencies and set up the OpenAI client.

In [ ]:
# Install dependencies
!pip install -q openai pydantic smolagents

In [1]:
import os
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from your environment

MODEL = "gpt-4.1-mini"  # using mini to keep costs low during the activity

# Quick test — if this prints a response, you're good to go!
test = client.chat.completions.create(
    model=MODEL,
    max_tokens=20,
    messages=[{"role": "user", "content": "Say 'hello' if you can hear me."}]
)
print("✅ Connected!", test.choices[0].message.content)

✅ Connected! Hello! I can see your message. How can I assist you today?


We'll also create a small helper so we don't repeat boilerplate in every exercise:

In [2]:
def chat(messages, temperature=0.0, max_tokens=1024, top_p=None, tools=None):
    """Send messages to the model and return the response."""
    kwargs = dict(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    if top_p is not None:
        kwargs["top_p"] = top_p
    if tools:
        kwargs["tools"] = tools
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message

---
## Exploring Model Parameters

Before we get into prompting techniques, let's play with the knobs that control **how** the model generates text. These parameters don't change *what* the model knows — they change how it picks its next word.

### The big three

| Parameter | What it does | Range |
|-----------|-------------|-------|
| **temperature** | Controls randomness. 0 = deterministic (always picks the most likely token). Higher = more random/creative. | 0.0 – 2.0 |
| **top_p** | Nucleus sampling. Only considers tokens whose cumulative probability adds up to `top_p`. Lower = fewer choices = more focused. | 0.0 – 1.0 |
| **top_k** | Only considers the top K most likely tokens. Not available in OpenAI's API, but used by other providers (Google, HuggingFace). | integer |

**Rule of thumb:** Adjust temperature *or* top_p, not both at the same time. They both control randomness in different ways and combining them can produce unpredictable results.

### Exercise 0a: Temperature

Let's ask the model the same creative question multiple times and see how temperature affects the output.

In [3]:
# Temperature = 0 — run this cell a few times. Notice the output is the SAME every time.
prompt = [{"role": "user", "content": "Suggest a creative name for an Italian restaurant"}]

for i in range(3):
    response = chat(prompt, temperature=0.0, max_tokens=50)
    print(f"  Run {i+1}: {response.content}")

  Run 1: Sure! How about **"La Tavola Viva"**? It means "The Living Table" in Italian, evoking a warm, lively atmosphere where friends and family gather to enjoy authentic Italian cuisine.
  Run 2: Sure! How about **"La Tavola Viva"**? It means "The Living Table" in Italian, evoking a warm, vibrant atmosphere where food and company come alive.
  Run 3: Sure! How about **"La Tavola Allegra"**? It means "The Joyful Table" in Italian, evoking a warm, inviting atmosphere perfect for enjoying delicious Italian cuisine.


In [4]:
# Temperature = 1.5 — same prompt, but now the outputs vary wildly
for i in range(3):
    response = chat(prompt, temperature=1.5, max_tokens=50)
    print(f"  Run {i+1}: {response.content}")

  Run 1: Sure! How about **"Trio e Vino"**?  
It combines the idea of a cozy trio of delicious dishes or close friends dining together, with the classic Italian love for wine, evoking warmth and authenticity. Let me know if
  Run 2: Sure! How about **"La Tavola Bella"**? 

It means "The Beautiful Table" in Italian, evoking the warmth of a welcoming, authentic Italian dining experience. Would you like suggestions with a particular style or theme?
  Run 3: Sure! How about **"La Tavola del Sogno"**? It means "The Table of Dreams" in Italian, evoking a warm, inviting place where delicious meals create unforgettable experiences. Let me know if you want more suggestions!


In [10]:
# TODO: Try a temperature between 0 and 2. What value feels like a good balance between creative and coherent?
response = chat(prompt, temperature=1, max_tokens=50)  # TODO: pick a value
print(response.content)

Sure! How about **"Bella Tavola"**? It means "Beautiful Table" in Italian, conveying a warm and inviting place to enjoy delicious Italian cuisine. Let me know if you'd like more options!


### Exercise 0b: Top-p (nucleus sampling)

Top-p works differently from temperature. Instead of scaling probabilities, it **limits which tokens the model can choose from**. A top_p of 0.1 means "only consider tokens in the top 10% of probability mass" — so the model picks from a very small set of likely words.

- `top_p=1.0` → consider all tokens (default)
- `top_p=0.1` → only the most likely tokens

In [11]:
# top_p = 0.1 — very focused, picks from a tiny pool of likely tokens
# We set temperature=1 so randomness comes from top_p, not temperature
for i in range(3):
    response = chat(prompt, temperature=1.0, top_p=0.1, max_tokens=50)
    print(f"  Run {i+1}: {response.content}")

  Run 1: Sure! How about **"La Tavola Viva"**? It means "The Living Table" in Italian, evoking a warm, lively atmosphere where friends and family gather to enjoy authentic Italian cuisine.
  Run 2: Sure! How about **"La Tavola Viva"**? It means "The Living Table" in Italian, evoking a warm, lively atmosphere where friends and family gather to enjoy authentic Italian cuisine.
  Run 3: Sure! How about **"La Tavola Viva"**? It means "The Living Table" in Italian, evoking a warm, vibrant atmosphere where food and company come alive.


In [12]:
# top_p = 1.0 — wide open, full vocabulary available
for i in range(3):
    response = chat(prompt, temperature=1.0, top_p=1.0, max_tokens=50)
    print(f"  Run {i+1}: {response.content}")

  Run 1: Sure! How about **"La Tavola Segreta"**? It means "The Secret Table" in Italian, evoking a cozy, inviting atmosphere with a hint of mystery and exclusivity. Perfect for an Italian restaurant! Would you like some
  Run 2: Sure! How about **"La Tavola Dorata"**? It means "The Golden Table" in Italian, evoking warmth, elegance, and a place where guests gather to enjoy delicious Italian cuisine.
  Run 3: Sure! How about **"Bella Tavola"**? It means "Beautiful Table" in Italian and evokes a warm, inviting atmosphere perfect for an Italian dining experience. If you'd like something different, feel free to let me know!


### When to use what?

| Scenario | Recommended settings |
|----------|---------------------|
| Classification, extraction, math | `temperature=0` — you want the single best answer |
| Creative writing, brainstorming | `temperature=0.7–1.2` — variety without nonsense |
| Code generation | `temperature=0–0.2` — correctness over creativity |
| Chatbot / conversation | `temperature=0.5–0.8` — natural but consistent |

**top_k** (not available in OpenAI, but used by Google Gemini and others) works like top_p but simpler — it just keeps the top K tokens by probability. For example, `top_k=40` means the model only picks from its 40 best guesses at each step.

For the rest of this notebook, we'll use `temperature=0` so our results are reproducible.

---
## 1. Zero-Shot Prompting

Zero-shot means we give the model a task with **no examples** — just a clear instruction.

### Exercise 1a: Classify a restaurant review

Write a `messages` list that asks the model to classify the sentiment of a restaurant review as **positive**, **negative**, or **mixed**.

Tips:
- Use a `system` message to define the model's role and output format
- Be specific — tell it to respond with exactly one word

In [15]:
review = "The pasta was absolutely divine but we waited 45 minutes for our table despite having a reservation."

messages = [
    {"role": "system", "content": "Classify the sentiment of restaurant reviews as exactly one word: positive, negative, or mixed."},
    {"role": "user", "content": f"Classify the sentiment of this review: {review}"}
]

response = chat(messages)
print(f"Review: {review}")
print(f"Sentiment: {response.content}")

Review: The pasta was absolutely divine but we waited 45 minutes for our table despite having a reservation.
Sentiment: mixed


### Exercise 1b: The system prompt matters

Now let's see how the **system prompt** changes behavior. Use the same review, but this time ask the model to respond with separate ratings for food and service (1-5 scale), one per line.

In [30]:
review = "The pasta was absolutely divine but we waited 45 minutes for our table despite having a reservation."

messages = [
    {"role": "system", "content": "Rate the food and service in restaurant reviews on a scale of 1-5. Respond with just 'Food: X/5' and 'Service: X/5', one per line."},
    {"role": "user", "content": review}
]

response = chat(messages)
print(response.content)

Food: 5/5  
Service: 2/5


### Exercise 1c: Structured Outputs

In 1b, we *asked* the model to use a specific text format — but it's just following instructions. It could still deviate, add extra text, or miss a field. **Structured Outputs** let you define a schema and the API *guarantees* the response conforms to it.

The easiest way to define a schema is with **Pydantic** — you just write a Python class and OpenAI handles the rest:

```python
from pydantic import BaseModel

class MyOutput(BaseModel):
    name: str
    score: int

response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[...],
    response_format=MyOutput
)

result = response.choices[0].message.parsed  # ← already a MyOutput object!
```

No raw JSON schemas, no manual parsing. You define a class, you get an object back.

In [24]:
from pydantic import BaseModel

review = "The pasta was absolutely divine but we waited 45 minutes for our table despite having a reservation."

class ReviewAnalysis(BaseModel):
    food_rating: int
    service_rating: int
    summary: str

# TODO: Define a Pydantic model called ReviewAnalysis with three fields:
#   - food_rating: int
#   - service_rating: int
#   - summary: str


# TODO: Use client.beta.chat.completions.parse() to get a structured response
#   - Pass response_format=ReviewAnalysis (your Pydantic class)
#   - Add a system message — just describe the task, no need to mention JSON
#   - Add a user message with the review

response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {
            "role": "system", 
            "content": "Analyze the restaurant review. Rate the food and service from 1 to 5 and provide a brief summary."
        },
        {
            "role": "user", 
            "content": review
        }
    ],
    response_format=ReviewAnalysis,
    temperature=0.0
)
        # TODO: Add a system message

        # TODO: Add the user message with the review

    
    # TODO: Pass your Pydantic model as response_format


# The response is already parsed into your Pydantic object — no json.loads needed!
result = response.choices[0].message.parsed
print(f"Food:    {result.food_rating}/5")
print(f"Service: {result.service_rating}/5")
print(f"Summary: {result.summary}")
print(f"\nType: {type(result)}")  # ← it's a ReviewAnalysis object, not a dict!

Food:    5/5
Service: 2/5
Summary: The food, particularly the pasta, was excellent and highly enjoyable. However, the service was lacking due to a long wait time despite having a reservation.

Type: <class '__main__.ReviewAnalysis'>


---
## 2. Few-Shot Prompting

Few-shot means we provide **examples** of the desired input → output behavior. The model learns the pattern from the examples and applies it to new input.

### Exercise 2: Extract structured data from reviews

We want to extract dishes mentioned in reviews and whether the comment about each dish is positive or negative. We'll teach the model the output format by providing two examples.

Fill in the `messages` list below. The pattern is:
1. System message
2. Example user message → Example assistant response
3. Example user message → Example assistant response
4. Actual user message (the one we want the model to respond to)

**Key takeaway:** Compare 1b vs 1c. In 1b, we had to carefully word the system prompt to get a specific format. In 1c, we define a schema and the API *guarantees* valid, typed output. In production, always prefer structured outputs when you need to parse the response programmatically.

In [25]:
messages = [
    {"role": "system", "content": "You extract dishes mentioned in restaurant reviews. For each dish, say whether the comment was positive or negative. List one dish per line in the format: - dish name: sentiment"},

    # Example 1 (provided for you)
    {"role": "user", "content": "The margherita pizza was crispy and fresh, but the tiramisu was stale."},
    {"role": "assistant", "content": "- margherita pizza: positive\n- tiramisu: negative"},

    # TODO: Add Example 2
    # Make up a review mentioning 2-3 dishes and write the expected output
    # User message: a review
    # Assistant message: the extraction in "- dish: sentiment" format
    {"role": "user", "content": "The grilled salmon was perfectly cooked and flavorful, but the Caesar salad was bland and the breadsticks were soggy."},
    {"role": "assistant", "content": "- grilled salmon: positive\n- Caesar salad: negative\n- breadsticks: negative"},

    # Actual review for the model to process
    {"role": "user", "content": "The sushi was incredibly fresh and the miso soup was comforting, but the tempura was way too oily."}
]

response = chat(messages)
print(response.content)

- sushi: positive
- miso soup: positive
- tempura: negative


**Discussion:** Compare the output format to your examples. Does the model follow the pattern you demonstrated?

---
## 3. Chain of Thought (CoT)

Chain of Thought prompting asks the model to **reason step by step** before giving a final answer. This helps with tasks that require logic or multi-step thinking.

### Exercise 3a: Recipe scaling with reasoning

A recipe serves 4. The user wants to serve 7. Ask the model to scale the ingredients, showing its work step by step.

Hint: Tell the model in the system prompt to think step by step before giving the final scaled recipe.

In [27]:
recipe = """
Classic Tomato Pasta (Serves 4):
- 400g spaghetti
- 2 cans (800g) crushed tomatoes
- 4 cloves garlic
- 1/3 cup olive oil
- 1 tsp red pepper flakes
- Fresh basil
- Salt and pepper to taste
"""

messages = [
    {"role": "system", "content": "You are a helpful assistant that scales recipes based on the number of servings. You will do this step by step with reason"},
    {"role": "user", "content": f"Scale this recipe from 4 servings to 7:\n{recipe}"}
    # TODO: Add a system message that instructs the model to think step by step

    # TODO: Add a user message asking to scale this recipe from 4 servings to 7

]

response = chat(messages, max_tokens=2048)
print(response.content)

Let's scale the recipe from 4 servings to 7 servings step by step.

Step 1: Determine the scaling factor.
- New servings = 7
- Original servings = 4
- Scaling factor = 7 / 4 = 1.75

Step 2: Multiply each ingredient quantity by the scaling factor.

- Spaghetti: 400g × 1.75 = 700g
- Crushed tomatoes: 800g × 1.75 = 1400g (or 1.4 kg)
- Garlic cloves: 4 × 1.75 = 7 cloves
- Olive oil: 1/3 cup × 1.75 = (1/3 × 1.75) cups = 0.583 cups (about 9.33 tablespoons)
- Red pepper flakes: 1 tsp × 1.75 = 1.75 tsp
- Fresh basil: Since this is to taste, increase roughly by 1.75 times or adjust as preferred.
- Salt and pepper: To taste, increase slightly or adjust as preferred.

Step 3: Present the scaled recipe.

Classic Tomato Pasta (Serves 7):
- 700g spaghetti
- 1.4 kg (about 1 and 3/4 cans if using 800g cans) crushed tomatoes
- 7 cloves garlic
- About 9 and 1/3 tablespoons (or 0.58 cups) olive oil
- 1 and 3/4 teaspoons red pepper flakes
- Fresh basil (adjust to taste)
- Salt and pepper to taste

Would y

### Exercise 3b: Without CoT (for comparison)

Now try the same task **without** chain-of-thought. Just ask for the scaled recipe directly. Compare the results — is the reasoning visible? Do you trust the numbers more or less?

In [28]:
messages = [
    {"role": "system", "content": "You are a cooking assistant. Scale this recipe from 4 servings to 7. Be concise and provide just the scaled ingredients without any explanation."},
    {"role": "user", "content": f"Scale this recipe from 4 servings to 7:\n{recipe}"}
    # TODO: Ask for the scaled recipe WITHOUT step-by-step reasoning
    #       (e.g., system message like "You are a cooking assistant. Be concise.")

]

response = chat(messages, max_tokens=2048)
print(response.content)

Classic Tomato Pasta (Serves 7):
- 700g spaghetti
- 3.5 cans (1400g) crushed tomatoes
- 7 cloves garlic
- 0.58 cup olive oil
- 1.75 tsp red pepper flakes
- Fresh basil
- Salt and pepper to taste


**Discussion:** Did the CoT version get the math right? Could you spot errors more easily when the reasoning was visible?

---
## 4. Retrieval-Augmented Generation (RAG)

RAG lets us give the model **knowledge it doesn't already have** by retrieving relevant documents and stuffing them into the prompt.

For this exercise, we have a small "knowledge base" of restaurant recipes. Instead of a full vector database, we'll use a simple keyword search to keep things focused on the prompting pattern.

### Step 1: Our knowledge base

Run the cell below to set up the recipe database and search function.

In [31]:
# Our "knowledge base" — a list of recipes as plain text strings.
# Each recipe is already formatted the way we'd want the model to read it.
# In a real app, these might be chunks of documents loaded from files or a database.

RECIPES = [
    """Recipe: Spaghetti Carbonara
Cuisine: Italian
Servings: 4
Prep time: 20 minutes
Ingredients: 400g spaghetti, 200g guanciale, 4 egg yolks, 100g pecorino romano, black pepper
Instructions: Cook pasta. Fry guanciale until crispy. Mix egg yolks with cheese. Toss hot pasta with guanciale, then quickly stir in egg mixture off heat. Season with pepper.
Allergens: gluten, eggs, dairy""",

    """Recipe: Pad Thai
Cuisine: Thai
Servings: 2
Prep time: 30 minutes
Ingredients: 200g rice noodles, 200g shrimp, 2 eggs, 100g bean sprouts, 3 tbsp fish sauce, 2 tbsp tamarind paste, 1 tbsp sugar, crushed peanuts, lime wedges
Instructions: Soak noodles. Stir-fry shrimp, push aside. Scramble eggs. Add noodles, sauce mixture, and bean sprouts. Toss together. Serve with peanuts and lime.
Allergens: shellfish, eggs, peanuts
Dietary: gluten-free""",

    """Recipe: Black Bean Tacos
Cuisine: Mexican
Servings: 4
Prep time: 15 minutes
Ingredients: 2 cans black beans, 8 corn tortillas, 1 avocado, salsa verde, pickled onions, cilantro, lime, cumin, chili powder
Instructions: Season and heat black beans with cumin and chili powder. Warm tortillas. Assemble with mashed avocado, beans, salsa verde, pickled onions, and cilantro.
Allergens: none
Dietary: vegan, gluten-free""",

    """Recipe: Chicken Tikka Masala
Cuisine: Indian
Servings: 4
Prep time: 45 minutes
Ingredients: 500g chicken breast, 200ml yogurt, 400ml tomato sauce, 200ml heavy cream, 2 tbsp garam masala, 1 tbsp turmeric, ginger, garlic, basmati rice
Instructions: Marinate chicken in yogurt and spices. Grill or bake chicken. Simmer tomato sauce with cream and spices. Add chicken to sauce. Serve over basmati rice.
Allergens: dairy
Dietary: gluten-free""",

    """Recipe: Mushroom Risotto
Cuisine: Italian
Servings: 4
Prep time: 40 minutes
Ingredients: 300g arborio rice, 250g mixed mushrooms, 1L vegetable broth, 1 onion, 100ml white wine, 50g parmesan, butter, thyme
Instructions: Sauté onion, add rice and toast. Deglaze with wine. Add broth one ladle at a time, stirring constantly. Fold in sautéed mushrooms, parmesan, and butter.
Allergens: dairy
Dietary: vegetarian""",
]


def search_recipes(query):
    """Search recipes by checking if any query word appears in the recipe text.
    In a real app, this would be vector similarity search over embeddings."""
    results = []
    for recipe in RECIPES:
        if any(word in recipe.lower() for word in query.lower().split()):
            results.append(recipe)
    return results


# Test the search
results = search_recipes("Italian")
print(f"Found {len(results)} result(s):\n")
print(results[0])  # show what a result looks like — just a string!
print(f"\n✅ Knowledge base loaded with {len(RECIPES)} recipes.")

Found 2 result(s):

Recipe: Spaghetti Carbonara
Cuisine: Italian
Servings: 4
Prep time: 20 minutes
Ingredients: 400g spaghetti, 200g guanciale, 4 egg yolks, 100g pecorino romano, black pepper
Instructions: Cook pasta. Fry guanciale until crispy. Mix egg yolks with cheese. Toss hot pasta with guanciale, then quickly stir in egg mixture off heat. Season with pepper.
Allergens: gluten, eggs, dairy

✅ Knowledge base loaded with 5 recipes.


### Step 2: Build the RAG prompt

The core idea of RAG is simple: **search your data, paste it into the prompt, then ask the model.**

Run the cell below to see exactly what the model will receive. Change the query to see how different questions retrieve different recipes.

In [36]:
# ✏️ Change this query to see how different questions pull in different recipes
query = "Any recipes I should avoid if I have a dairy allergy?"

# Step 1: Search — find relevant recipes
results = search_recipes(query)
print(f"📚 Search complete")

# Step 2: Build the full prompt — this is EXACTLY what the model will see
system_message = f"""You are a restaurant recipe assistant. Answer the user's question based ONLY on the provided context. If the context does not contain enough information to answer, say "I don't have that information in our recipe database."

Context:
{results}"""

print("=" * 60)
print("SYSTEM MESSAGE (what the model receives):")
print("=" * 60)
print(system_message)
print("=" * 60)
print(f"\nUSER MESSAGE: {query}")
print("=" * 60)

📚 Search complete
SYSTEM MESSAGE (what the model receives):
You are a restaurant recipe assistant. Answer the user's question based ONLY on the provided context. If the context does not contain enough information to answer, say "I don't have that information in our recipe database."

Context:
['Recipe: Spaghetti Carbonara\nCuisine: Italian\nServings: 4\nPrep time: 20 minutes\nIngredients: 400g spaghetti, 200g guanciale, 4 egg yolks, 100g pecorino romano, black pepper\nInstructions: Cook pasta. Fry guanciale until crispy. Mix egg yolks with cheese. Toss hot pasta with guanciale, then quickly stir in egg mixture off heat. Season with pepper.\nAllergens: gluten, eggs, dairy', 'Recipe: Pad Thai\nCuisine: Thai\nServings: 2\nPrep time: 30 minutes\nIngredients: 200g rice noodles, 200g shrimp, 2 eggs, 100g bean sprouts, 3 tbsp fish sauce, 2 tbsp tamarind paste, 1 tbsp sugar, crushed peanuts, lime wedges\nInstructions: Soak noodles. Stir-fry shrimp, push aside. Scramble eggs. Add noodles, sauce

### Step 3: Send it to the model

Now send the prompt you just built. The `system_message` and `query` variables carry over from the cell above.

In [37]:
# Send the prompt we built above to the model
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": query}
]

response = chat(messages)
print(response.content)

If you have a dairy allergy, you should avoid the following recipes from the list:

- Spaghetti Carbonara (contains pecorino romano cheese)
- Chicken Tikka Masala (contains yogurt and heavy cream)
- Mushroom Risotto (contains parmesan and butter)

These recipes contain dairy ingredients that could trigger an allergy.


In [ ]:
# Try again with a query that should fail — we don't have desserts
# Go back to the cell above, change the query to "How do I make chocolate lava cake?"
# and re-run both cells. Does the model hallucinate or stay grounded?

**Discussion:** Try different queries in the cell above — "dairy allergy", "vegan", "chocolate lava cake". Look at the printed system message each time. Notice how the *same code* produces completely different prompts depending on what the search returns. That's the power of RAG.

---
## 5. Tool Use & Agents with smolagents

So far, the model can only *generate text*. But what if it could **take actions** — search a database, calculate something, call an API?

**Tool use** lets the model decide which functions to call. An **agent** is the loop that keeps calling tools until the task is done.

We'll use [smolagents](https://github.com/huggingface/smolagents), a lightweight agent framework from Hugging Face. It handles all the plumbing — tool call parsing, message management, the agent loop — so we can focus on defining tools and watching the agent work.

### Step 1: Set up the model

smolagents can use any OpenAI-compatible model. We'll point it at the same API key you've been using.

In [39]:
from smolagents import tool, ToolCallingAgent, OpenAIModel

model = OpenAIModel(
    model_id="gpt-4.1-mini",
    api_key=os.environ["OPENAI_API_KEY"],
)

print("✅ smolagents model ready")

✅ smolagents model ready


### Step 2: Define tools

In smolagents, a tool is just a Python function with the `@tool` decorator. The framework reads the function's **name**, **type hints**, and **docstring** to tell the model what the tool does and how to call it.

This is how you give an LLM the ability to take actions — you write normal Python functions, and the model decides when to use them.

In [40]:
# Tool 1: Search our recipe knowledge base (reuses RECIPES from section 4)
@tool
def search_recipes(query: str) -> str:
    """Search the restaurant recipe database by keyword. Returns matching recipes with ingredients, instructions, and allergen info.

    Args:
        query: Search keywords, e.g. 'vegan gluten-free' or 'Italian pasta'
    """
    results = []
    for recipe in RECIPES:
        if any(word in recipe.lower() for word in query.lower().split()):
            results.append(recipe)
    if results:
        return "\n\n".join(results)
    return "No matching recipes found."


# Tool 2: Nutrition calculator
@tool
def calculate_nutrition(recipe_name: str) -> str:
    """Estimate the calories and macros per serving for a recipe.

    Args:
        recipe_name: The exact name of the recipe, e.g. 'Spaghetti Carbonara'
    """
    nutrition_data = {
        "Spaghetti Carbonara": {"calories": 650, "protein": "25g", "carbs": "70g", "fat": "30g"},
        "Pad Thai": {"calories": 450, "protein": "20g", "carbs": "55g", "fat": "15g"},
        "Black Bean Tacos": {"calories": 380, "protein": "15g", "carbs": "50g", "fat": "12g"},
        "Chicken Tikka Masala": {"calories": 550, "protein": "35g", "carbs": "40g", "fat": "25g"},
        "Mushroom Risotto": {"calories": 480, "protein": "12g", "carbs": "65g", "fat": "18g"},
    }
    info = nutrition_data.get(recipe_name)
    if info:
        return f"{recipe_name}: {info['calories']} cal, {info['protein']} protein, {info['carbs']} carbs, {info['fat']} fat per serving"
    return f"No nutrition data found for '{recipe_name}'"


print("✅ Tools defined: search_recipes, calculate_nutrition")
print(f"\nsearch_recipes description: {search_recipes.description}")
print(f"calculate_nutrition description: {calculate_nutrition.description}")

✅ Tools defined: search_recipes, calculate_nutrition

search_recipes description: Search the restaurant recipe database by keyword. Returns matching recipes with ingredients, instructions, and allergen info.
calculate_nutrition description: Estimate the calories and macros per serving for a recipe.


### Step 3: Create the agent

A `ToolCallingAgent` ties everything together — you give it a model and a list of tools, and it handles the rest. When you ask it a question, it decides which tools to call, calls them, reads the results, and keeps going until it has a complete answer.

Watch the terminal output — smolagents logs every step so you can see the agent's thinking in real time.

In [41]:
# TODO: Create a ToolCallingAgent
#   - Pass model=model
#   - Pass tools=[search_recipes, calculate_nutrition]

agent = ToolCallingAgent(
    model=model,
    tools=[search_recipes, calculate_nutrition],
    # TODO: fill in the arguments
)

print("✅ Agent ready! It has access to:", agent.tools)

✅ Agent ready! It has access to: {'search_recipes': <smolagents.tools.tool.<locals>.SimpleTool object at 0x734bc69a6600>, 'calculate_nutrition': <smolagents.tools.tool.<locals>.SimpleTool object at 0x734bc69a6f00>, 'final_answer': <smolagents.default_tools.FinalAnswerTool object at 0x734bcd253cb0>}


### Step 4: Test the agent

Now let's ask the agent questions. Watch the logs — you'll see it decide which tools to call, execute them, and use the results to form an answer.

**Simple query** — should trigger one tool call:

In [42]:
# Simple — one tool call
result = agent.run("What Italian dishes do you have?")
print("\n📋 Final answer:", result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What Italian dishes do you have?                                                                                │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'search_recipes' with arguments: {'query': 'Italian'}                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Recipe: Spaghetti Carbonara
Cuisine: Italian
Servings: 4
Prep time: 20 minutes
Ingredients: 400g spaghetti, 200g guanciale, 4 egg yolks, 100g pecorino romano, black pepper
Instructions: Cook pasta. Fry guanciale until crispy. Mix egg yolks with cheese. Toss hot pasta with guanciale, 
then quickly stir in egg mixture off heat. Season with pepper.
Allergens: gluten, eggs, dairy

Recipe: Mushroom Risotto
Cuisine: Italian
Servings: 4
Prep time: 40 minutes
Ingredients: 300g arborio rice, 250g mixed mushrooms, 1L vegetable broth, 1 onion, 100ml white wine, 50g parmesan, 
butter, thyme
Instructions: Sauté onion, add rice and toast. Deglaze with wine. Add broth one ladle at a time, stirring 
constantly. Fold in sautéed mushrooms, parmesan, and butter.
Allergens: dairy
Dietary: vegetarian

[Step 1: Duration 0.69 seconds| Input tokens: 1,077 | Output tokens: 15]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The Italian dishes available are:\n1. Spaghetti        │
│ Carbonara - Ingredients include spaghetti, guanciale, egg yolks, pecorino romano, and black pepper. Contains    │
│ allergens: gluten, eggs, dairy.\n2. Mushroom Risotto - Ingredients include arborio rice, mixed mushrooms,       │
│ vegetable broth, onion, white wine, parmesan, butter, and thyme. It is a vegetarian dish and contains dairy     │
│ allergen.'}                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The Italian dishes available are:
1. Spaghetti Carbonara - Ingredients include spaghetti, guanciale, egg yolks, pecorino romano, and black pepper. 
Contains allergens: gluten, eggs, dairy.
2. Mushroom Risotto - Ingredients include arborio rice, mixed mushrooms, vegetable broth, onion, white wine, 
parmesan, butter, and thyme. It is a vegetarian dish and contains dairy allergen.

Final answer: The Italian dishes available are:
1. Spaghetti Carbonara - Ingredients include spaghetti, guanciale, egg yolks, pecorino romano, and black pepper. 
Contains allergens: gluten, eggs, dairy.
2. Mushroom Risotto - Ingredients include arborio rice, mixed mushrooms, vegetable broth, onion, white wine, 
parmesan, butter, and thyme. It is a vegetarian dish and contains dairy allergen.

[Step 2: Duration 1.55 seconds| Input tokens: 2,436 | Output tokens: 114]


📋 Final answer: The Italian dishes available are:
1. Spaghetti Carbonara - Ingredients include spaghetti, guanciale, egg yolks, pecorino romano, and black pepper. Contains allergens: gluten, eggs, dairy.
2. Mushroom Risotto - Ingredients include arborio rice, mixed mushrooms, vegetable broth, onion, white wine, parmesan, butter, and thyme. It is a vegetarian dish and contains dairy allergen.


**Multi-step query** — the agent needs to search first, then look up nutrition. Watch how it chains the tools together:

In [43]:
# Multi-step — requires search + nutrition lookup
result = agent.run("I want something gluten-free with under 500 calories. What do you recommend?")
print("\n📋 Final answer:", result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ I want something gluten-free with under 500 calories. What do you recommend?                                    │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'search_recipes' with arguments: {'query': 'gluten-free under 500 calories'}                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Recipe: Pad Thai
Cuisine: Thai
Servings: 2
Prep time: 30 minutes
Ingredients: 200g rice noodles, 200g shrimp, 2 eggs, 100g bean sprouts, 3 tbsp fish sauce, 2 tbsp tamarind paste, 1
tbsp sugar, crushed peanuts, lime wedges
Instructions: Soak noodles. Stir-fry shrimp, push aside. Scramble eggs. Add noodles, sauce mixture, and bean 
sprouts. Toss together. Serve with peanuts and lime.
Allergens: shellfish, eggs, peanuts
Dietary: gluten-free

Recipe: Black Bean Tacos
Cuisine: Mexican
Servings: 4
Prep time: 15 minutes
Ingredients: 2 cans black beans, 8 corn tortillas, 1 avocado, salsa verde, pickled onions, cilantro, lime, cumin, 
chili powder
Instructions: Season and heat black beans with cumin and chili powder. Warm tortillas. Assemble with mashed 
avocado, beans, salsa verde, pickled onions, and cilantro.
Allergens: none
Dietary: vegan, gluten-free

Recipe: Chicken Tikka Masala
Cuisine: Indian
Servings: 4
Prep time: 45 minutes
Ingredients: 500g chicken breast, 200ml yogurt, 400ml tomato sauce, 200ml heavy cream, 2 tbsp garam masala, 1 tbsp 
turmeric, ginger, garlic, basmati rice
Instructions: Marinate chicken in yogurt and spices. Grill or bake chicken. Simmer tomato sauce with cream and 
spices. Add chicken to sauce. Serve over basmati rice.
Allergens: dairy
Dietary: gluten-free

[Step 1: Duration 0.76 seconds| Input tokens: 1,086 | Output tokens: 21]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'I recommend the Black Bean Tacos or Pad Thai as        │
│ gluten-free options under 500 calories. Black Bean Tacos have 380 calories per serving and Pad Thai has 450     │
│ calories per serving.'}                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: I recommend the Black Bean Tacos or Pad Thai as gluten-free options under 500 calories. Black Bean 
Tacos have 380 calories per serving and Pad Thai has 450 calories per serving.

Final answer: I recommend the Black Bean Tacos or Pad Thai as gluten-free options under 500 calories. Black Bean 
Tacos have 380 calories per serving and Pad Thai has 450 calories per serving.

[Step 3: Duration 1.20 seconds| Input tokens: 4,335 | Output tokens: 148]


📋 Final answer: I recommend the Black Bean Tacos or Pad Thai as gluten-free options under 500 calories. Black Bean Tacos have 380 calories per serving and Pad Thai has 450 calories per serving.


In [44]:
# Complex — reasoning + multiple tools
result = agent.run(
    "I'm planning dinner for a group. One person is vegan, another is allergic to dairy. "
    "Can you find options that work for both and tell me the nutrition info?"
)
print("\n📋 Final answer:", result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ I'm planning dinner for a group. One person is vegan, another is allergic to dairy. Can you find options that   │
│ work for both and tell me the nutrition info?                                                                   │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'search_recipes' with arguments: {'query': 'vegan dairy-free'}                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Recipe: Black Bean Tacos
Cuisine: Mexican
Servings: 4
Prep time: 15 minutes
Ingredients: 2 cans black beans, 8 corn tortillas, 1 avocado, salsa verde, pickled onions, cilantro, lime, cumin, 
chili powder
Instructions: Season and heat black beans with cumin and chili powder. Warm tortillas. Assemble with mashed 
avocado, beans, salsa verde, pickled onions, and cilantro.
Allergens: none
Dietary: vegan, gluten-free

[Step 1: Duration 0.74 seconds| Input tokens: 1,103 | Output tokens: 18]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The Black Bean Tacos recipe is suitable for both vegan │
│ and dairy-free diets. The nutritional information per serving is approximately 380 calories, 15g protein, 50g   │
│ carbs, and 12g fat.'}                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The Black Bean Tacos recipe is suitable for both vegan and dairy-free diets. The nutritional 
information per serving is approximately 380 calories, 15g protein, 50g carbs, and 12g fat.

Final answer: The Black Bean Tacos recipe is suitable for both vegan and dairy-free diets. The nutritional 
information per serving is approximately 380 calories, 15g protein, 50g carbs, and 12g fat.

[Step 3: Duration 0.82 seconds| Input tokens: 3,741 | Output tokens: 94]


📋 Final answer: The Black Bean Tacos recipe is suitable for both vegan and dairy-free diets. The nutritional information per serving is approximately 380 calories, 15g protein, 50g carbs, and 12g fat.


### What just happened?

Look back at the logs above. The agent followed a **think → act → observe** loop:

1. **Think** — the model decided which tool to call and why
2. **Act** — it called the tool with specific arguments
3. **Observe** — it read the result and decided what to do next
4. **Repeat** — it kept going until it had enough information to answer

This is the same loop behind ChatGPT plugins, GitHub Copilot, and every other AI agent. The only difference is scale — more tools, more complex reasoning, more steps.

**This is the full ladder in action:**
- The system prompt uses **zero-shot** instruction
- The agent **reasons** about what to do (chain of thought)
- It **retrieves** recipes from a knowledge base (RAG)
- It **calls tools** to get nutrition data (tool use)
- It does all of this **autonomously** in a loop (agentic AI)

---
## Recap

| Section | What you built | Key concept |
|---------|---------------|-------------|
| 1. Zero-Shot | Review classifier | Clear instructions, no examples |
| 2. Few-Shot | Review data extractor | Teaching by demonstration |
| 3. Chain of Thought | Recipe scaler | Step-by-step reasoning |
| 4. RAG | Recipe Q&A assistant | Grounding in external knowledge |
| 5. Tool Use & Agents | Restaurant agent with search + nutrition | Model as autonomous decision-maker |

Each technique built on the one before it. In real applications, you'll combine them — an agent that uses few-shot prompts, chain-of-thought reasoning, retrieval, and tool calling all at once.